# MCVR CIFAR-10 GPU smoke test
Auto-generated by `scripts/build_kaggle_notebook.py` from commit `436b152d9eb9fae5aaf71aad355d16febc41b082` of Monte-Carlo-variance-reduction-for-diffusion-model-training. Do not edit by hand.

In [ ]:
%cd /kaggle/working
!mkdir -p src/mcvr scripts configs/plans results/raw logs data
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q clean-fid==0.1.35 2>&1 | tail -1
import torch, torchvision, sys; print(sys.version, torch.__version__, torchvision.__version__)

In [ ]:
%%writefile src/mcvr/__init__.py
"""Monte Carlo variance reduction for DDPM training."""


In [ ]:
%%writefile src/mcvr/unet.py
"""Small DDPM U-Net (Ho et al. 2020 style), kept deliberately compact."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


def gn(c):
    return nn.GroupNorm(min(32, c // 4), c)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim, dropout=0.0):
        super().__init__()
        self.n1, self.c1 = gn(cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.t = nn.Linear(tdim, cout)
        self.n2, self.c2 = gn(cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.t(F.silu(temb))[:, :, None, None]
        h = self.c2(self.drop(F.silu(self.n2(h))))
        return h + self.skip(x)


class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n, self.qkv, self.o = gn(c), nn.Conv2d(c, 3 * c, 1), nn.Conv2d(c, c, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        q, k, v = self.qkv(self.n(x)).reshape(b, 3, c, h * w).unbind(1)
        a = F.scaled_dot_product_attention(q.transpose(1, 2)[:, None], k.transpose(1, 2)[:, None],
                                           v.transpose(1, 2)[:, None])[:, 0]
        return x + self.o(a.transpose(1, 2).reshape(b, c, h, w))


class UNet(nn.Module):
    def __init__(self, in_ch=3, ch=64, mult=(1, 2, 2), nres=2, attn_lowest=True, dropout=0.0):
        super().__init__()
        self.ch = ch
        tdim = ch * 4
        self.temb = nn.Sequential(nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.inp = nn.Conv2d(in_ch, ch, 3, padding=1)
        self.down = nn.ModuleList()
        chans, c = [ch], ch
        for i, m in enumerate(mult):
            for _ in range(nres):
                self.down.append(ResBlock(c, ch * m, tdim, dropout)); c = ch * m; chans.append(c)
            if i < len(mult) - 1:
                self.down.append(nn.Conv2d(c, c, 3, stride=2, padding=1)); chans.append(c)
        self.mid1 = ResBlock(c, c, tdim, dropout)
        self.mida = Attn(c) if attn_lowest else nn.Identity()
        self.mid2 = ResBlock(c, c, tdim, dropout)
        self.up = nn.ModuleList()
        for i, m in reversed(list(enumerate(mult))):
            for _ in range(nres + 1):
                self.up.append(ResBlock(c + chans.pop(), ch * m, tdim, dropout)); c = ch * m
            if i > 0:
                self.up.append(nn.Upsample(scale_factor=2, mode="nearest"))
                self.up.append(nn.Conv2d(c, c, 3, padding=1))
        self.out = nn.Sequential(gn(c), nn.SiLU(), nn.Conv2d(c, in_ch, 3, padding=1))

    def forward(self, x, t):
        temb = self.temb(timestep_embedding(t, self.ch))
        h = self.inp(x)
        hs = [h]
        for m in self.down:
            h = m(h, temb) if isinstance(m, ResBlock) else m(h)
            hs.append(h)
        h = self.mid2(self.mida(self.mid1(h, temb)), temb)
        for m in self.up:
            if isinstance(m, ResBlock):
                h = m(torch.cat([h, hs.pop()], 1), temb)
            else:
                h = m(h)
        return self.out(h)


In [ ]:
%%writefile src/mcvr/diffusion.py
"""DDPM forward process, per-sample L_simple loss and a deterministic DDIM sampler."""
import torch


class Diffusion:
    def __init__(self, T=1000, beta1=1e-4, betaT=0.02, device="cpu"):
        self.T = T
        betas = torch.linspace(beta1, betaT, T, dtype=torch.float64)
        abar = torch.cumprod(1 - betas, 0)
        self.sqrt_abar = abar.sqrt().float().to(device)
        self.sqrt_1m_abar = (1 - abar).sqrt().float().to(device)
        self.abar = abar.float().to(device)

    def q_sample(self, x0, t, eps):
        return self.sqrt_abar[t][:, None, None, None] * x0 + self.sqrt_1m_abar[t][:, None, None, None] * eps

    def per_sample_loss(self, model, x0, t, eps):
        """Unweighted L_simple per sample: mean over pixels of (eps_hat - eps)^2."""
        pred = model(self.q_sample(x0, t, eps), t)
        return (pred.float() - eps).pow(2).flatten(1).mean(1)

    @torch.no_grad()
    def ddim_sample(self, model, n, shape, steps=50, generator=None, device="cpu", batch=1000, amp=False):
        """Deterministic DDIM (eta=0). Initial noise comes from `generator` so that every model
        is sampled from the *same* noise (common random numbers)."""
        ts = torch.linspace(self.T - 1, 0, steps, dtype=torch.float64).round().long().tolist()
        out = []
        for i0 in range(0, n, batch):
            b = min(batch, n - i0)
            x = torch.randn((b,) + tuple(shape), generator=generator, device="cpu").to(device)
            for j, t in enumerate(ts):
                tt = torch.full((b,), t, device=device, dtype=torch.long)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                    eps = model(x, tt).float()
                a = self.abar[t]
                x0 = ((x - (1 - a).sqrt() * eps) / a.sqrt()).clamp(-1, 1)
                eps = (x - a.sqrt() * x0) / (1 - a).sqrt()  # re-derive eps after clipping
                if j + 1 < len(ts):
                    an = self.abar[ts[j + 1]]
                    x = an.sqrt() * x0 + (1 - an).sqrt() * eps
                else:
                    x = x0
            out.append(x.cpu())
        return torch.cat(out)


In [ ]:
%%writefile src/mcvr/estimators.py
"""Monte Carlo estimators of the DDPM L_simple objective.

Every estimator returns, for a budget of n (x0, t, eps) evaluations:
    idx  : LongTensor[n]  indices into the training set (repeated for paired estimators)
    flip : BoolTensor[n]  horizontal-flip flags, drawn per distinct x0 (repeated for pairs)
    t    : LongTensor[n]  timesteps in {0..T-1}
    eps  : FloatTensor[n, C, H, W]
    w    : FloatTensor[n] importance weights (1 for unweighted estimators)
and the loss estimate is mean(w * per_sample_loss).  All are unbiased for
E_{x0 ~ data, t ~ U{0..T-1}, eps ~ N(0,I)} [ per_sample_loss ].
"""
import torch


class ISState:
    """Running per-bin estimate of E[L_t^2] (Improved-DDPM rule: p(t) ∝ sqrt(E[L_t^2])).

    The proposal is piecewise constant over `K` equal bins of timesteps and mixed with the
    uniform distribution with weight `mix` so that importance weights are bounded by 1/mix.
    It is uniform until every bin holds `history` losses (warm-up, as in Improved DDPM)."""

    def __init__(self, T, K=50, history=100, mix=0.1, device="cpu"):
        assert T % K == 0
        self.T, self.K, self.H, self.mix = T, K, history, mix
        self.buf = torch.zeros(K, history, device=device)
        self.cnt = torch.zeros(K, dtype=torch.long, device=device)
        self.warm = False  # becomes True once every bin holds `history` losses (checked until then)

    @torch.no_grad()
    def update(self, t, loss):
        """Vectorised ring-buffer update with the squared per-sample (unweighted) losses."""
        b = (t // (self.T // self.K)).long()
        order = torch.argsort(b, stable=True)
        bs, v = b[order], loss.detach().float()[order] ** 2
        counts = torch.bincount(bs, minlength=self.K)
        starts = torch.cumsum(counts, 0) - counts
        rank = torch.arange(len(bs), device=bs.device) - starts[bs]
        pos = (self.cnt[bs] + rank) % self.H
        self.buf[bs, pos] = v
        self.cnt += counts

    def probs(self):
        if not self.warm:
            self.warm = bool((self.cnt >= self.H).all())
        if not self.warm:
            return torch.full((self.K,), 1.0 / self.K, device=self.buf.device)
        p = self.buf.mean(1).sqrt()
        p = p / p.sum()
        return (1 - self.mix) * p + self.mix / self.K

    def state_dict(self):
        return {"buf": self.buf, "cnt": self.cnt}

    def load_state_dict(self, d):
        self.buf.copy_(d["buf"]); self.cnt.copy_(d["cnt"]); self.warm = False


def _stratified_u(m, gen, device):
    return (torch.arange(m, device=device) + torch.rand(m, generator=gen, device=device)) / m


def _u_to_t(u, T):
    return (u * T).long().clamp_(0, T - 1)


class Estimator:
    name = "base"
    paired = False  # True -> n/2 distinct (x0, t), each used twice

    def __init__(self, T, n_data, shape, device, is_state=None, seed=0):
        self.T, self.n_data, self.shape, self.device = T, n_data, tuple(shape), device
        self.is_state = is_state

    def _t(self, m, gen):
        return torch.randint(0, self.T, (m,), generator=gen, device=self.device)

    def _eps(self, m, gen):
        return torch.randn((m,) + self.shape, generator=gen, device=self.device)

    def _idx(self, m, gen):
        idx = torch.randint(0, self.n_data, (m,), generator=gen, device=self.device)
        flip = torch.rand(m, generator=gen, device=self.device) < 0.5
        return idx, flip

    def sample(self, n, gen):
        (idx, flip), t, eps = self._idx(n, gen), self._t(n, gen), self._eps(n, gen)
        return idx, flip, t, eps, torch.ones(n, device=self.device)


class Uniform(Estimator):
    name = "uniform"


class Stratified(Estimator):
    name = "stratified"

    def _t(self, m, gen):
        return _u_to_t(_stratified_u(m, gen, self.device), self.T)


class Sobol(Estimator):
    """Consecutive points of a scrambled 1-D Sobol sequence (torch SobolEngine: random linear
    matrix scramble + digital shift).  With n a power of two each batch is a (0, k, 1)-net."""
    name = "sobol"

    def __init__(self, *a, seed=0, **k):
        super().__init__(*a, seed=seed, **k)
        self.engine = torch.quasirandom.SobolEngine(1, scramble=True, seed=seed + 12345)

    def _t(self, m, gen):
        return _u_to_t(self.engine.draw(m)[:, 0].to(self.device), self.T)


class Importance(Estimator):
    name = "importance"

    def _sample_t(self, u):
        p = self.is_state.probs()
        K, wbin = self.is_state.K, self.T // self.is_state.K
        cdf = torch.cumsum(p, 0)
        cdf[-1] = 1.0
        b = torch.searchsorted(cdf, u, right=True).clamp_(0, K - 1)
        lo = cdf[b] - p[b]
        pos = ((u - lo) / p[b]).clamp_(0, 1 - 1e-6)  # position inside the bin, uniform
        t = b * wbin + (pos * wbin).long()
        w = 1.0 / (K * p[b])
        return t.clamp_(0, self.T - 1), w

    def sample(self, n, gen):
        u = torch.rand(n, generator=gen, device=self.device)
        t, w = self._sample_t(u)
        return (*self._idx(n, gen), t, self._eps(n, gen), w)


class ISStrat(Importance):
    """Stratified uniforms pushed through the IS inverse CDF (cf. CARV)."""
    name = "is_strat"

    def sample(self, n, gen):
        t, w = self._sample_t(_stratified_u(n, gen, self.device))
        return (*self._idx(n, gen), t, self._eps(n, gen), w)


class Antithetic(Estimator):
    name = "antithetic"
    paired = True

    def _pair_t(self, m, gen):
        return self._t(m, gen)

    def _second_eps(self, eps, gen):
        return -eps

    def sample(self, n, gen):
        assert n % 2 == 0
        m = n // 2
        (idx, flip), t, eps = self._idx(m, gen), self._pair_t(m, gen), self._eps(m, gen)
        return (idx.repeat(2), flip.repeat(2), t.repeat(2),
                torch.cat([eps, self._second_eps(eps, gen)]), torch.ones(n, device=self.device))


class PairedIID(Antithetic):
    """Diagnostic: same (x0, t) pairing as antithetic but two *independent* eps."""
    name = "paired_iid"

    def _second_eps(self, eps, gen):
        return torch.randn(eps.shape, generator=gen, device=self.device)


class StratAnti(Antithetic):
    name = "strat_anti"

    def _pair_t(self, m, gen):
        return _u_to_t(_stratified_u(m, gen, self.device), self.T)


REGISTRY = {c.name: c for c in [Uniform, Stratified, Sobol, Importance, ISStrat, Antithetic, PairedIID, StratAnti]}


def make(name, T, n_data, shape, device, is_state, seed=0):
    return REGISTRY[name](T, n_data, shape, device, is_state=is_state, seed=seed)


In [ ]:
%%writefile src/mcvr/data.py
"""Datasets held entirely on the training device as uint8 tensors (fast random indexing)."""
import gzip
import os
import numpy as np
import torch


def _fmnist(root, split):
    pre = "train" if split == "train" else "t10k"
    with gzip.open(os.path.join(root, f"{pre}-images-idx3-ubyte.gz"), "rb") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 1, 28, 28)
    return torch.from_numpy(x.copy())


def _cifar10(root, split):
    import torchvision
    ds = torchvision.datasets.CIFAR10(root, train=(split == "train"), download=True)
    return torch.from_numpy(ds.data).permute(0, 3, 1, 2).contiguous()


def load(name, root, split, device):
    x = {"fmnist": _fmnist, "cifar10": _cifar10}[name](root, split)
    return x.to(device)


class ImageData:
    """uint8 images on device; `get(idx, flip)` returns float images in [-1, 1].

    Random horizontal flips (if enabled) are drawn per *distinct* x0 by the caller so that an
    antithetic pair shares exactly the same x0."""

    def __init__(self, x, hflip):
        self.x, self.hflip = x, hflip
        self.n, self.shape = x.shape[0], tuple(x.shape[1:])

    def get(self, idx, flip=None):
        x = self.x[idx].float().div_(127.5).sub_(1.0)
        if self.hflip and flip is not None:
            x = torch.where(flip[:, None, None, None], x.flip(3), x)
        return x


In [ ]:
%%writefile src/mcvr/probes.py
"""Direct measurement of gradient-estimator variance at fixed parameters, and a
law-of-total-variance decomposition of the per-sample gradient by timestep."""
import torch

from . import estimators as E


def flat_grad(model):
    return torch.cat([p.grad.reshape(-1) for p in model.parameters()])


def estimator_grad(model, diff, data, idx, flip, t, eps, w, amp, scale=1024.0):
    """Gradient of mean(w * L_i) at the current parameters, as one flat fp32 vector.
    Under fp16 autocast a fixed loss scale is used and removed (no optimiser state touched)."""
    model.zero_grad(set_to_none=True)
    x0 = data.get(idx, flip)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
        loss = (w * diff.per_sample_loss(model, x0, t, eps)).mean()
    (loss * (scale if amp else 1.0)).backward()
    g = flat_grad(model)
    if amp:
        g = g / scale
    return g, loss.detach()


def variance_probe(model, diff, pool, names, is_state, n, R, seed, amp):
    """For each estimator: R independent draws of the n-sample gradient estimator at the same
    parameters.  x0 is drawn from the fixed probe pool `pool` (an ImageData).  Returns trace of
    covariance, squared norm of the mean, and squared distance of each mean to the uniform mean
    (unbiasedness check: E||mean_e - mean_u||^2 = (tr_e + tr_u)/R if both are unbiased)."""
    was_training = model.training
    model.eval()  # no dropout in our configs; eval() only for safety
    dev = pool.x.device
    out, means = {}, {}
    for name in names:
        gen = torch.Generator(device=dev); gen.manual_seed(seed)
        est = E.make(name, diff.T, pool.n, pool.shape, dev, is_state, seed=seed)
        S, SS, nonfinite = None, 0.0, 0
        for r in range(R):
            g, _ = estimator_grad(model, diff, pool, *est.sample(n, gen), amp)
            if not torch.isfinite(g).all():
                nonfinite += 1; continue
            g64 = g.double()
            S = g64.clone() if S is None else S.add_(g64)
            SS += float(g64.pow(2).sum())
        Rv = R - nonfinite
        mean = S / Rv
        tr = (SS - Rv * float(mean.pow(2).sum())) / (Rv - 1)
        means[name] = mean
        out[name] = {"trace_cov": tr, "mean_sqnorm": float(mean.pow(2).sum()), "R": Rv, "nonfinite": nonfinite}
    if "uniform" in means:
        for name in names:
            d = float((means[name] - means["uniform"]).pow(2).sum())
            out[name]["dist2_to_uniform_mean"] = d
            out[name]["dist2_expected_if_unbiased"] = (out[name]["trace_cov"] + out["uniform"]["trace_cov"]) / out[name]["R"] if name != "uniform" else 0.0
    model.zero_grad(set_to_none=True)
    model.train(was_training)
    return out


def timestep_decomposition(model, diff, pool, K, n, reps, seed, amp, loss_samples=512):
    """Per timestep bin k (K equal bins of {0..T-1}):
      m_k     : E[g | t in bin k]                            (from two independent batches A, B)
      Vw_k    : per-sample within-bin variance  = n ||gA - gB||^2 / 2
      Veps_k  : per-sample E[Var(g | x0, t)]    = n ||gA - gC||^2 / 2   (C: same x0,t, new eps)
      Vanti_k : (1 - rho_k) Veps_k              = n ||gA - gA'||^2 / 2  (A': same x0,t, -eps)
    plus per-sample loss mean / variance in the bin (forward only).
    Returns per-bin scalars and estimator-level variance predictions for N = 2n... (computed later)."""
    was_training = model.training
    model.eval()
    dev = pool.x.device
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    T, wbin = diff.T, diff.T // K
    rows, mks = [], []
    for k in range(K):
        acc = {"vw": 0.0, "veps": 0.0, "vanti": 0.0}
        msum = None
        for _ in range(reps):
            def draw():
                idx = torch.randint(0, pool.n, (n,), generator=gen, device=dev)
                flip = torch.rand(n, generator=gen, device=dev) < 0.5
                t = k * wbin + torch.randint(0, wbin, (n,), generator=gen, device=dev)
                eps = torch.randn((n,) + pool.shape, generator=gen, device=dev)
                return idx, flip, t, eps
            one = torch.ones(n, device=dev)
            ia, fa, ta, ea = draw()
            gA, _ = estimator_grad(model, diff, pool, ia, fa, ta, ea, one, amp)
            gAm, _ = estimator_grad(model, diff, pool, ia, fa, ta, -ea, one, amp)
            eC = torch.randn(ea.shape, generator=gen, device=dev)
            gC, _ = estimator_grad(model, diff, pool, ia, fa, ta, eC, one, amp)
            gB, _ = estimator_grad(model, diff, pool, *draw(), one, amp)
            acc["vw"] += n * float((gA - gB).double().pow(2).sum()) / 2
            acc["veps"] += n * float((gA - gC).double().pow(2).sum()) / 2
            acc["vanti"] += n * float((gA - gAm).double().pow(2).sum()) / 2
            m = (gA.double() + gB.double()) / 2
            msum = m if msum is None else msum + m
        mk = msum / reps
        mks.append(mk)
        vw, veps, vanti = acc["vw"] / reps, acc["veps"] / reps, acc["vanti"] / reps
        # per-sample loss statistics (no grad)
        with torch.no_grad():
            Ls = []
            for i0 in range(0, loss_samples, n):
                idx = torch.randint(0, pool.n, (n,), generator=gen, device=dev)
                flip = torch.rand(n, generator=gen, device=dev) < 0.5
                t = k * wbin + torch.randint(0, wbin, (n,), generator=gen, device=dev)
                eps = torch.randn((n,) + pool.shape, generator=gen, device=dev)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                    Ls.append(diff.per_sample_loss(model, pool.get(idx, flip), t, eps).float())
            L = torch.cat(Ls)
        rows.append({"bin": k, "t_lo": k * wbin, "t_hi": (k + 1) * wbin - 1,
                     "Vw": vw, "Veps": veps, "Vanti": vanti, "rho": 1 - vanti / veps,
                     # var of mk estimate is Vw/(2 n reps); subtract it from ||mk||^2
                     "mk_sqnorm": float(mk.pow(2).sum()) - vw / (2 * n * reps),
                     "loss_mean": float(L.mean()), "loss_var": float(L.var()),
                     "loss2_mean": float(L.pow(2).mean())})
    M = torch.stack(mks)
    mbar = M.mean(0)
    noise = sum(r["Vw"] for r in rows) / len(rows) / (2 * n * reps)
    between = float((M - mbar).pow(2).sum(1).mean()) - noise * (1 - 1 / K)
    model.zero_grad(set_to_none=True)
    model.train(was_training)
    return {"bins": rows, "V_between": between, "mbar_sqnorm": float(mbar.pow(2).sum()),
            "K": K, "n": n, "reps": reps}


def predict_variances(dec, N, is_probs=None):
    """Predicted tr Cov of each N-sample estimator from the decomposition (per-sample units / N).
    Uses bin-level quantities only, so 'stratified' is the bin-stratified approximation
    (within-bin t variation is kept in Vw)."""
    bins = dec["bins"]; K = len(bins)
    Vw = torch.tensor([b["Vw"] for b in bins]); Veps = torch.tensor([b["Veps"] for b in bins])
    rho = torch.tensor([b["rho"] for b in bins]); m2 = torch.tensor([b["mk_sqnorm"] for b in bins])
    Vb, mbar2 = dec["V_between"], dec["mbar_sqnorm"]
    Vx0t = Vw - Veps
    pred = {"uniform": (Vw.mean() + Vb) / N,
            "stratified": Vw.mean() / N,
            "paired_iid": (2 * (Vb + Vx0t.mean()) + Veps.mean()) / N,
            "antithetic": (2 * (Vb + Vx0t.mean()) + ((1 + rho) * Veps).mean()) / N,
            "strat_anti": (2 * Vx0t.mean() + ((1 + rho) * Veps).mean()) / N}
    second = Vw + m2  # E||g_i||^2 per bin
    q_opt = second.clamp(min=0).sqrt(); q_opt = q_opt / q_opt.sum()
    pred["oracle_grad_IS"] = ((second / (K ** 2 * q_opt)).sum() - mbar2) / N
    if is_probs is not None:
        q = torch.as_tensor(is_probs, dtype=torch.float64).reshape(K, -1).sum(1)
        pred["importance"] = ((second.double() / (K ** 2 * q)).sum() - mbar2) / N
    return {k: float(v) for k, v in pred.items()}


In [ ]:
%%writefile src/mcvr/train.py
"""Train one DDPM with a given (x0, t, eps) estimator.  Resumable from out_dir/ckpt.pt.

Usage: python -m mcvr.train --config configs/cifar10.yaml --set estimator=antithetic seed=1
"""
import argparse
import copy
import json
import os
import platform
import subprocess
import time

import torch
import yaml

from . import data as D
from . import estimators as E
from . import probes as P
from .diffusion import Diffusion
from .unet import UNet


def load_config(path, overrides=()):
    with open(path) as f:
        cfg = yaml.safe_load(f)
    for o in overrides:
        k, v = o.split("=", 1)
        cfg[k] = yaml.safe_load(v)
    cfg.setdefault("run_id", f"{cfg['dataset']}_{cfg['estimator']}_s{cfg['seed']}")
    cfg["out_dir"] = os.path.join(cfg["out_dir"], cfg["run_id"])
    return cfg


def jdump(path, obj):
    with open(path, "a") as f:
        f.write(json.dumps(obj) + "\n")


class Timer:
    """Wall-clock of optimiser steps only (evaluation/probe time is excluded)."""

    def __init__(self, cuda):
        self.cuda, self.total = cuda, 0.0

    def sync(self):
        if self.cuda:
            torch.cuda.synchronize()

    def start(self):
        self.sync(); self.t0 = time.perf_counter()

    def stop(self):
        self.sync(); self.total += time.perf_counter() - self.t0


@torch.no_grad()
def heldout_loss(model, diff, test, crn, amp, bs=500):
    """Held-out L_simple with common random numbers: identical (x0, t, eps) for every model."""
    model.eval()
    tot = 0.0
    for i in range(0, len(crn["t"]), bs):
        sl = slice(i, i + bs)
        x0 = test.get(crn["idx"][sl])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            tot += float(diff.per_sample_loss(model, x0, crn["t"][sl], crn["eps"][sl]).sum())
    model.train()
    return tot / len(crn["t"])


def run(cfg):
    out = cfg["out_dir"]
    os.makedirs(out, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() and cfg.get("device", "auto") != "cpu" else "cpu")
    cuda = dev.type == "cuda"
    amp = bool(cfg.get("amp", True)) and cuda
    torch.backends.cudnn.benchmark = True
    if not cuda and cfg.get("threads"):
        torch.set_num_threads(int(cfg["threads"]))

    # ---- data -------------------------------------------------------------------------
    train_x = D.load(cfg["dataset"], cfg["data_root"], "train", dev)
    test_x = D.load(cfg["dataset"], cfg["data_root"], "test", dev)
    train = D.ImageData(train_x, cfg.get("hflip", False))
    test = D.ImageData(test_x, False)
    g0 = torch.Generator(device="cpu"); g0.manual_seed(10_000)  # fixed across all runs (CRN)
    pool_idx = torch.randperm(train.n, generator=g0)[: cfg.get("probe_pool", 4096)].to(dev)
    pool = D.ImageData(train_x[pool_idx], cfg.get("hflip", False))
    n_ho = min(cfg.get("heldout_n", 2000), test.n)
    crn = {"idx": torch.randperm(test.n, generator=g0)[:n_ho].to(dev),
           "t": torch.randint(0, cfg["T"], (n_ho,), generator=g0).to(dev),
           "eps": torch.randn((n_ho,) + train.shape, generator=g0).to(dev)}

    # ---- model ------------------------------------------------------------------------
    torch.manual_seed(cfg["seed"])  # identical init for a given seed across estimators
    model = UNet(in_ch=train.shape[0], **cfg["model"]).to(dev)
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    diff = Diffusion(cfg["T"], device=dev)
    is_state = E.ISState(cfg["T"], K=cfg.get("is_bins", 50), history=cfg.get("is_history", 100),
                         mix=cfg.get("is_mix", 0.1), device=dev)
    est = E.make(cfg["estimator"], cfg["T"], train.n, train.shape, dev, is_state, seed=cfg["seed"])
    gen = torch.Generator(device=dev); gen.manual_seed(cfg["seed"] * 1_000_003 + 7)

    step, timer = 0, Timer(cuda)
    ck = os.path.join(out, "ckpt.pt")
    if os.path.exists(ck):
        s = torch.load(ck, map_location=dev, weights_only=False)
        model.load_state_dict(s["model"]); ema.load_state_dict(s["ema"]); opt.load_state_dict(s["opt"])
        scaler.load_state_dict(s["scaler"]); is_state.load_state_dict(s["is_state"])
        gen.set_state(s["gen"].cpu()); step = s["step"]; timer.total = s["train_time"]
        if hasattr(est, "engine"):
            est.engine.fast_forward(step * cfg["batch"])
        print(f"[{cfg['run_id']}] resumed at step {step}")
    else:
        meta = {"cfg": cfg, "n_params": n_params, "device": str(dev),
                "gpu": torch.cuda.get_device_name(0) if cuda else platform.processor() or "cpu",
                "torch": torch.__version__, "python": platform.python_version(),
                "start_time": time.strftime("%Y-%m-%dT%H:%M:%S")}
        try:
            meta["git"] = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
        except Exception:
            meta["git"] = cfg.get("git", "unknown")
        json.dump(meta, open(os.path.join(out, "meta.json"), "w"), indent=1)

    def save():
        torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(),
                    "scaler": scaler.state_dict(), "is_state": is_state.state_dict(),
                    "gen": gen.get_state(), "step": step, "train_time": timer.total}, ck + ".tmp")
        os.replace(ck + ".tmp", ck)

    def evaluate(step):
        rec = {"step": step, "train_time": timer.total,
               "heldout": heldout_loss(model, diff, test, crn, amp),
               "heldout_ema": heldout_loss(ema, diff, test, crn, amp)}
        jdump(os.path.join(out, "evals.jsonl"), rec)
        print(f"[{cfg['run_id']}] step {step} heldout {rec['heldout']:.5f} ema {rec['heldout_ema']:.5f} t={timer.total:.0f}s", flush=True)

    probe_at, fid_at = set(cfg.get("probe_at", [])), set(cfg.get("fid_at", []))
    decomp_at = set(cfg.get("decomp_at", []))
    eval_every, log_every, ck_every = cfg.get("eval_every", 500), cfg.get("log_every", 50), cfg.get("ckpt_every", 1000)

    def hooks(step):
        if step % eval_every == 0 or step == cfg["steps"]:
            evaluate(step)
        if step in probe_at:
            t0 = time.perf_counter()
            res = P.variance_probe(model, diff, pool, cfg["probe_estimators"], is_state, cfg["batch"],
                                   cfg["probe_R"], seed=50_000 + step, amp=amp)
            jdump(os.path.join(out, "probes.jsonl"), {"step": step, "res": res, "is_probs": is_state.probs().tolist(),
                                                      "seconds": time.perf_counter() - t0})
            base = res["uniform"]["trace_cov"]
            print(f"[{cfg['run_id']}] probe@{step}: " + " ".join(f"{k}={v['trace_cov'] / base:.3f}" for k, v in res.items()), flush=True)
        if step in decomp_at:
            t0 = time.perf_counter()
            dec = P.timestep_decomposition(model, diff, pool, cfg.get("decomp_bins", 20), cfg["batch"] // 2,
                                           cfg.get("decomp_reps", 2), seed=60_000 + step, amp=amp)
            dec["pred"] = P.predict_variances(dec, cfg["batch"], None)
            jdump(os.path.join(out, "decomp.jsonl"), {"step": step, "dec": dec, "seconds": time.perf_counter() - t0})
        if step in fid_at:
            from .fid import fid_eval
            t0 = time.perf_counter()
            r = fid_eval(ema, diff, cfg, dev, amp, os.path.join(out, f"samples_{step}.png"))
            r.update(step=step, train_time=timer.total, seconds=time.perf_counter() - t0)
            jdump(os.path.join(out, "fid.jsonl"), r)
            print(f"[{cfg['run_id']}] FID@{step} = {r['fid']:.2f} ({r['seconds']:.0f}s)", flush=True)

    def boundary(step):
        return (step % log_every == 0 or step % eval_every == 0 or step % ck_every == 0 or step == cfg["steps"]
                or step in probe_at or step in fid_at or step in decomp_at)

    if step == 0:
        hooks(0)
    model.train()
    timer.start()
    while step < cfg["steps"]:
        idx, flip, t, eps, w = est.sample(cfg["batch"], gen)
        x0 = train.get(idx, flip)
        lr = cfg["lr"] * min(1.0, (step + 1) / cfg.get("warmup", 200))
        for gparam in opt.param_groups:
            gparam["lr"] = lr
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            L = diff.per_sample_loss(model, x0, t, eps)
            loss = (w * L).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.get("clip", 1.0))
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            torch._foreach_lerp_(list(ema.parameters()), list(model.parameters()), 1 - cfg.get("ema", 0.999))
        is_state.update(t, L)  # every run tracks the IS proposal (used by probes; sampled only by IS)
        step += 1
        if boundary(step):
            timer.stop()  # synchronises; only optimiser-step time is accumulated
            if step % log_every == 0:
                jdump(os.path.join(out, "train.jsonl"), {"step": step, "loss": float(loss.detach()), "unweighted": float(L.detach().mean()),
                                                         "gnorm": float(gnorm), "train_time": timer.total})
            hooks(step)
            if step % ck_every == 0 or step == cfg["steps"]:
                save()
            timer.start()
    json.dump({"finished": True, "step": step, "train_time": timer.total,
               "ms_per_step": 1000 * timer.total / step, "end_time": time.strftime("%Y-%m-%dT%H:%M:%S")},
              open(os.path.join(out, "done.json"), "w"))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--set", nargs="*", default=[])
    a = ap.parse_args()
    run(load_config(a.config, a.set))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/mcvr/fid.py
"""FID with clean-fid (Parmar et al. 2022), `clean` mode, against CIFAR-10 *train* statistics.

All models are sampled from the same initial noise (generator seeded with `fid_noise_seed`)
using deterministic DDIM, so FID differences are not inflated by sampler noise."""
import numpy as np
import torch


def save_grid(x, path, nrow=8):
    from PIL import Image
    x = ((x[: nrow * nrow].clamp(-1, 1) + 1) * 127.5).round().byte().permute(0, 2, 3, 1).numpy()
    n, h, w, c = x.shape
    g = x.reshape(nrow, nrow, h, w, c).transpose(0, 2, 1, 3, 4).reshape(nrow * h, nrow * w, c)
    Image.fromarray(g.squeeze(-1) if c == 1 else g).save(path)


def fid_eval(model, diff, cfg, dev, amp, grid_path=None):
    from cleanfid import fid
    gen = torch.Generator(device="cpu"); gen.manual_seed(cfg.get("fid_noise_seed", 2024))
    n = cfg.get("fid_n", 10000)
    x = diff.ddim_sample(model, n, (3, 32, 32), steps=cfg.get("ddim_steps", 50), generator=gen,
                         device=dev, batch=cfg.get("fid_batch", 1000), amp=amp)
    if grid_path:
        save_grid(x, grid_path)
    imgs = ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)
    state = {"i": 0}

    def gen_fn(z):  # clean-fid asks for batches; we serve the pre-generated samples in order
        b = z.shape[0]
        out = imgs[state["i"]: state["i"] + b]
        state["i"] += b
        return out

    score = fid.compute_fid(gen=gen_fn, dataset_name="cifar10", dataset_res=32, dataset_split="train",
                            mode="clean", num_gen=n, batch_size=500, z_dim=1, device=dev, verbose=False)
    return {"fid": float(score), "n": n, "ddim_steps": cfg.get("ddim_steps", 50)}


In [ ]:
%%writefile scripts/run_queue.py
"""Run a list of training jobs, one sequential queue per GPU (or a single CPU queue).

plan JSON: {"runs": [{"config": "configs/cifar10.yaml", "set": ["estimator=uniform", "seed=0"]}, ...]}
Jobs whose out_dir already contains done.json are skipped; unfinished jobs resume from ckpt.pt.
No new job is started after --deadline seconds (a guard on the compute budget)."""
import argparse
import json
import os
import subprocess
import sys
import threading
import time

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from mcvr.train import load_config  # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--plan", required=True)
ap.add_argument("--gpus", default="")  # e.g. "0,1"; empty -> CPU
ap.add_argument("--deadline", type=float, default=1e9)
ap.add_argument("--extra", nargs="*", default=[])
a = ap.parse_args()

runs = json.load(open(a.plan))["runs"]
T0 = time.time()
slots = a.gpus.split(",") if a.gpus else ["cpu"]
queues = [runs[i::len(slots)] for i in range(len(slots))]
os.makedirs("logs", exist_ok=True)


def worker(slot, q):
    for r in q:
        sets = r["set"] + a.extra
        cfg = load_config(r["config"], sets)
        out = cfg["out_dir"]
        if os.path.exists(os.path.join(out, "done.json")):
            print(f"[{slot}] skip {cfg['run_id']} (done)", flush=True); continue
        if time.time() - T0 > a.deadline:
            print(f"[{slot}] deadline reached, not starting {cfg['run_id']}", flush=True); continue
        env = dict(os.environ, PYTHONPATH=os.path.join(os.getcwd(), "src"))
        if slot != "cpu":
            env["CUDA_VISIBLE_DEVICES"] = slot
        cmd = [sys.executable, "-m", "mcvr.train", "--config", r["config"], "--set", *sets]
        t0 = time.time()
        with open(os.path.join("logs", cfg["run_id"] + ".log"), "a") as lf:
            rc = subprocess.call(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)
        print(f"[{slot}] {cfg['run_id']} rc={rc} {time.time() - t0:.0f}s", flush=True)
        os.makedirs(out, exist_ok=True)
        with open(os.path.join(out, "status.json"), "w") as f:
            json.dump({"rc": rc, "wall_s": time.time() - t0}, f)


ths = [threading.Thread(target=worker, args=(s, q)) for s, q in zip(slots, queues)]
[t.start() for t in ths]
[t.join() for t in ths]
print(f"queue finished in {time.time() - T0:.0f}s")


In [ ]:
%%writefile configs/cifar10.yaml
# CIFAR-10 main configuration (Kaggle GPU). Only `estimator` and `seed` vary between runs.
dataset: cifar10
data_root: data/cifar10
estimator: uniform
seed: 0
T: 1000
model: {ch: 64, mult: [1, 2, 2], nres: 2, attn_lowest: true, dropout: 0.0}
batch: 128
lr: 2.0e-4
warmup: 200
clip: 1.0
ema: 0.999
hflip: true
amp: true
steps: 6000
eval_every: 500
log_every: 50
ckpt_every: 1000
heldout_n: 2000
probe_pool: 4096
probe_at: []          # set per run in the plan (uniform runs only)
probe_estimators: [uniform, antithetic, paired_iid, stratified, sobol, importance, is_strat, strat_anti]
probe_R: 48
decomp_at: []
decomp_bins: 20
decomp_reps: 2
fid_at: [2000, 4000, 6000]
fid_n: 10000
fid_batch: 1000
ddim_steps: 50
fid_noise_seed: 2024
out_dir: results/raw/cifar10


In [ ]:
%%writefile configs/plans/cifar10_smoke.json
{
 "runs": [
  {
   "config": "configs/cifar10.yaml",
   "set": [
    "estimator=uniform",
    "seed=0",
    "steps=400",
    "eval_every=200",
    "probe_at=[400]",
    "decomp_at=[400]",
    "fid_at=[400]",
    "probe_R=16",
    "run_id=smoke_uniform_s0",
    "out_dir=results/raw/smoke"
   ]
  },
  {
   "config": "configs/cifar10.yaml",
   "set": [
    "estimator=importance",
    "seed=0",
    "steps=400",
    "eval_every=200",
    "fid_at=[]",
    "run_id=smoke_importance_s0",
    "out_dir=results/raw/smoke"
   ]
  },
  {
   "config": "configs/cifar10.yaml",
   "set": [
    "estimator=strat_anti",
    "seed=0",
    "steps=400",
    "eval_every=200",
    "fid_at=[]",
    "run_id=smoke_strat_anti_s0",
    "out_dir=results/raw/smoke"
   ]
  }
 ]
}

In [ ]:
# Resume support: if a previous version of this notebook is attached as an input, copy its results
import glob, shutil, os
for d in glob.glob('/kaggle/input/*/results/raw/*/*'):
    dst = d.split('/results/', 1)[1]; dst = os.path.join('results', dst)
    if not os.path.exists(dst): shutil.copytree(d, dst); print('restored', dst)


In [ ]:
!echo '{"git": "436b152d9eb9fae5aaf71aad355d16febc41b082"}' > results/kaggle_git.json
!python scripts/run_queue.py --plan configs/plans/cifar10_smoke.json --gpus $(python -c "import torch;print(','.join(map(str,range(torch.cuda.device_count()))))") --deadline 900.0 --extra git=436b152d9eb9fae5aaf71aad355d16febc41b082

In [ ]:
!tail -n 5 logs/*.log
!tar czf results_small.tgz --exclude='*.pt' results logs && ls -la results_small.tgz